# Airport Supply Analysis

This notebook evaluates airport demand-supply mismatch and post-airport-trip
captain behavior to determine whether targeted acquisition of
airport-catchment captains is the right intervention.

In [1]:
import pandas as pd

In [4]:
airport_hourly = pd.read_csv(
    "../Data/airport_hourly.csv",
    parse_dates=["hour_ts"]
)

print("Airport hourly shape:", airport_hourly.shape)

Airport hourly shape: (10248, 9)


In [5]:
airport_trips = pd.read_csv(
    "../Data/airport_trips.csv"
)

print("Airport trips shape:", airport_trips.shape)

Airport trips shape: (60000, 9)


In [9]:
print("Airport hourly columns:")
print(airport_hourly.columns.tolist())

Airport hourly columns:
['zone_id', 'zone_type', 'hour_ts', 'requests', 'fulfilled_requests', 'unfulfilled_requests', 'online_captains', 'avg_eta_min', 'avg_surge_multiplier']


In [10]:
print("Airport trips columns:")
print(airport_trips.columns.tolist())

Airport trips columns:
['trip_id', 'pickup_zone_id', 'drop_zone_id', 'drop_zone_type', 'request_ts', 'trip_distance_km', 'captain_cancelled', 'got_return_fare_within_20min', 'fare_inr']


## 1. Airport Demand-Supply Mismatch

Airport terminals are evaluated for the concentration and timing of
unfulfilled demand, online captain supply, ETA, and surge.

In [11]:
airport_hourly["unfulfilled_rate"] = (
    airport_hourly["unfulfilled_requests"]
    / airport_hourly["requests"]
)

terminal_hourly = airport_hourly[
    airport_hourly["zone_type"] == "airport_terminal"
].copy()

print("Terminal rows:", len(terminal_hourly))

print("\nOverall terminal metrics:")
print(
    terminal_hourly[
        [
            "requests",
            "fulfilled_requests",
            "unfulfilled_requests",
            "online_captains",
            "avg_eta_min",
            "avg_surge_multiplier"
        ]
    ].sum(numeric_only=True)
)

Terminal rows: 2928

Overall terminal metrics:
requests                136814.00
fulfilled_requests       81756.00
unfulfilled_requests     55058.00
online_captains          87214.00
avg_eta_min              16834.66
avg_surge_multiplier      4116.81
dtype: float64


In [12]:
print("\nTerminal unfulfilled rate:")
print(
    round(
        terminal_hourly["unfulfilled_requests"].sum()
        / terminal_hourly["requests"].sum()
        * 100,
        2
    ),
    "%"
)

print("\nAverage online captains:",
      round(terminal_hourly["online_captains"].mean(), 2))

print("Average ETA:",
      round(terminal_hourly["avg_eta_min"].mean(), 2))

print("Average surge:",
      round(terminal_hourly["avg_surge_multiplier"].mean(), 2))


Terminal unfulfilled rate:
40.24 %

Average online captains: 29.79
Average ETA: 5.75
Average surge: 1.41


In [13]:
hourly_pressure = (
    terminal_hourly
    .groupby(terminal_hourly["hour_ts"].dt.hour)
    .agg(
        requests=("requests", "sum"),
        unfulfilled=("unfulfilled_requests", "sum"),
        online_captains=("online_captains", "mean"),
        avg_eta=("avg_eta_min", "mean"),
        avg_surge=("avg_surge_multiplier", "mean")
    )
)

hourly_pressure["unfulfilled_rate"] = (
    hourly_pressure["unfulfilled"]
    / hourly_pressure["requests"]
)

print(
    hourly_pressure
    .sort_values("unfulfilled_rate", ascending=False)
    .round(2)
)

         requests  unfulfilled  online_captains  avg_eta  avg_surge  \
hour_ts                                                               
23          13007         9594            12.57    10.04       2.17   
1           11288         8145            12.84     9.83       2.15   
22          11188         8042            13.02     9.90       2.14   
0            9124         6191            12.60     9.54       2.07   
2            9500         6427            13.21     9.58       2.06   
21           7293         4347            13.34     8.77       1.93   
3            6190         3274            13.22     8.01       1.80   
4            4718         1499            14.98     6.18       1.47   
6            6776         1944            21.84     5.88       1.43   
5            5291         1452            17.58     5.83       1.41   
20           4346         1159            15.65     5.45       1.38   
7            7459         1189            29.72     4.76       1.23   
19    

## 2. Post-Airport Trip Behavior

We examine airport-origin trips by drop-zone type to assess captain
cancellation, return-fare availability within 20 minutes, and trip economics.

In [14]:
trip_summary = (
    airport_trips
    .groupby("drop_zone_type")
    .agg(
        trips=("trip_id", "count"),
        cancellation_rate=("captain_cancelled", "mean"),
        return_fare_rate=("got_return_fare_within_20min", "mean"),
        avg_distance_km=("trip_distance_km", "mean"),
        avg_fare_inr=("fare_inr", "mean")
    )
)

trip_summary["cancellation_rate"] *= 100
trip_summary["return_fare_rate"] *= 100

print(trip_summary.round(2))

                trips  cancellation_rate  return_fare_rate  avg_distance_km  \
drop_zone_type                                                                
city_core       25282               8.63             52.72            13.24   
suburban        24643              21.00             16.53            23.38   
tech_park       10075               8.96             41.43            16.10   

                avg_fare_inr  
drop_zone_type                
city_core             216.62  
suburban              353.57  
tech_park             255.23  


In [15]:
print("\nOverall airport-trip behavior:")

print(
    "Captain cancellation:",
    round(airport_trips["captain_cancelled"].mean() * 100, 2),
    "%"
)

print(
    "Return fare within 20 min:",
    round(
        airport_trips["got_return_fare_within_20min"].mean() * 100,
        2
    ),
    "%"
)


Overall airport-trip behavior:
Captain cancellation: 13.77 %
Return fare within 20 min: 35.96 %


## 3. Intervention Decision

The airport has a clear supply-demand mismatch, concentrated during 21:00–03:00.
However, post-trip behavior suggests that broad acquisition of airport-catchment
captains is not the most direct intervention.

Suburban airport-origin trips have the highest captain cancellation rate (21.00%)
and lowest return-fare availability within 20 minutes (16.53%). This suggests
that post-trip economics and repositioning may affect captain willingness to
serve airport demand.

Recommendation:
Prioritize a targeted airport supply intervention during 21:00–03:00 focused
on improving captain economics and return-fare availability rather than broad
airport-catchment acquisition.

A randomized or controlled pilot should measure incremental fulfilled requests,
captain participation, cancellation, and incentive cost.

Limitation:
The dataset contains only airport-origin trips, so airport cancellation and
return-fare rates cannot be compared against non-airport trips.

## 4. Ranked Recommendations

### 1. Fix RC verification friction
The DL→RC stage is the largest funnel leak, with 5,489 captains lost.
RC also has the highest verification-failure volume. Prioritize better
document capture guidance, pre-submit image-quality checks, and retake
prompts.

Illustrative scenario: recovering 10% of RC drop-offs would recover ~549
captains; at the current ~27.3% downstream approval rate, this corresponds
to ~150 additional approvals. This is a scenario, not a causal forecast.

Success metric: RC clearance rate and signup→approved conversion.
Risk: image-quality interventions will not address all document failures.

### 2. Improve Fitness→Insurance completion
The Fitness→Insurance stage loses 3,340 captains and shows meaningful
variation across acquisition and device segments. Prioritize targeted
nudges and assisted completion for low-performing segments.

Illustrative scenario: recovering 10% of these drop-offs would recover ~334
captains; at ~90.6% Insurance→Approval conversion, this corresponds to
~303 additional approvals. This is a scenario, not a causal forecast.

Success metric: Fitness→Insurance conversion.
Risk: intervention may require additional assisted-support cost.

### 3. Target airport supply during 21:00–03:00
Airport terminals account for ~86% of all unfulfilled requests, with the
largest mismatch concentrated between 21:00 and 03:00.

Prioritize a targeted supply/repositioning incentive pilot rather than broad
airport-catchment captain acquisition.

Impact should be measured experimentally using incremental fulfilled requests
per incremental captain-hour and cost per incremental fulfilled request.

Success metrics: unfulfilled rate, fulfilled requests, online captain-hours,
cancellation rate, and intervention cost.

Risk: incentives may shift existing supply from other zones rather than add
net supply.